# iTransformer 5min — 實驗平台

**只需修改 Config，其餘全自動執行。**

In [ ]:
# ══════════════════════════════════════════
#  SETUP — 只需執行一次（Runtime 重啟後再跑）
# ══════════════════════════════════════════
import os, sys

# 1. Clone or pull latest code
if not os.path.exists('/content/itransformer_5min'):
    os.system('git clone https://github.com/wiwihaun/itransformer_5min.git')
else:
    os.system('git -C /content/itransformer_5min pull')

os.chdir('/content/itransformer_5min')

# 2. Install dependencies
os.system('pip install -q patool sktime statsmodels reformer_pytorch')

# 3. Paths
for p in ['/content/itransformer_5min', '/content/itransformer_5min/toolbywi']:
    if p not in sys.path: sys.path.insert(0, p)

print("✅ Setup 完成！")


## ⚙️ Config — 在這裡調整所有參數

In [ ]:
# ══════════════════════════════════════════════════════
#  CONFIG — 唯一需要修改的地方
#  TP/SL 自動同步訓練目標標籤 & 回測止損止盈，無需重複設定
# ══════════════════════════════════════════════════════
config = {
    # ── 資料 ──────────────────────────────────────────
    'symbol':        'BTCUSDT',
    'alt_symbols':   ['ETHUSDT', 'BNBUSDT', 'SOLUSDT'],  # 空 list = 不用 alt
    'start_year':    2025,  'start_month':  6,
    'end_year':      2026,  'end_month':    3,

    # ── 目標標籤 ──────────────────────────────────────
    'direction':     'long',   # 'long' or 'short'
    'tp_pct':        0.02,     # 2% 止盈（訓練+回測統一）
    'sl_pct':        0.01,     # 1% 止損（訓練+回測統一）
    'lookahead':     288,      # 未來 288 根 = 24h

    # ── 模型架構 ──────────────────────────────────────
    'seq_len':       576,
    'd_model':       256,
    'd_ff':          512,
    'n_heads':       8,
    'e_layers':      2,
    'dropout':       0.2,

    # ── 訓練 ──────────────────────────────────────────
    'batch_size':    64,
    'learning_rate': 0.0001,
    'lradj':         'type3',
    'patience':      15,
    'train_epochs':  100,

    # ── 回測 ──────────────────────────────────────────
    'prob_threshold':  0.60,
    'position_size':   3,
    'max_hold_days':   1,
    'cooldown_hours':  4,
    'initial_capital': 1000.0,
}

print(f"Config 設定完成 ✔  方向={config['direction']}  TP={config['tp_pct']*100:.0f}%  SL={config['sl_pct']*100:.0f}%")


## 🚀 執行實驗

In [ ]:
# ══════════════════════════════════════════
#  RUN — 一鍵執行：下載 → 特徵 → 訓練 → 回測
# ══════════════════════════════════════════
from importlib import reload
import experiment_runner as _er
reload(_er)
from experiment_runner import run_experiment

results = run_experiment(config)


## 📊 結果顯示

In [ ]:
# ══════════════════════════════════════════
#  DISPLAY — 統計摘要 + 回測圖表
# ══════════════════════════════════════════
from experiment_runner import display_results
display_results(results)


## 💾 儲存模型（選用）

In [ ]:
# ══════════════════════════════════════════
#  SAVE — 儲存最佳 checkpoint 到 Google Drive
# ══════════════════════════════════════════
from experiment_runner import save_best_model
save_best_model(results, name=results['exp_id'])
